# 气泡图（东中西部对比）

> 对应代码：`MathStatsCode/code_in_notes/Chap.7/bubble_chart.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：以气泡大小代表人口，展示城市人均GDP与工业废水排放量的关系，并按东/中/西部分色。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.7`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.7")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

清空内存并载入城市数据（含年份、城市代码、GDP、人口、工业废水排放等变量）。


In [ ]:
%%stata
clear
use ../datasets/citydata.dta
keep if Year==2011


**省份代码、判断东中西部**

从城市代码中提取省份代码（`floor(CityCode/10000)`），再用 `inlist()` 生成三个地区虚拟变量：直辖市和东部沿海省份取 east=1，中部省份取 middle=1，西部省份取 west=1。`inlist(x,a,b,...)` 判断 x 是否等于列表中的任一值，是构造地区哑变量的简洁写法。


In [ ]:
%%stata
gen prov=floor(CityCode/10000)
gen east=inlist(prov,11,12,13,21,31,32,33,35,37,44,46)
gen middle=inlist(prov,14,22,23,34,36,41,43,42)
gen west=inlist(prov,15,45,50,51,52,53,54,61,62,63,64,65)


**产生变量**

构造画图所需的三个变量：`log_gdp_per_capita=log(v84)-log(v86)` 利用对数差等于比值对数（人均GDP=GDP/人口）；`log_pop=log(v86)` 为人口对数；`log_waste_water=log(v266)` 为工业废水排放量对数。


In [ ]:
%%stata
gen log_gdp_per_capita=log(v84)-log(v86)
gen log_pop=log(v86)
gen log_waste_water=log(v266)


**画图**

`twoway` 气泡图：三个 `scatter` 图层分别画东、中、西部城市，`[fw=v86]` 指定**频数权重**——权重变量决定每个点（气泡）的大小，因此人口越多的城市气泡越大（这是“气泡图”的核心）；`msize(tiny)` 控制基础大小、`mcolor(blue%30)` 表示蓝色且30%不透明度（半透明便于看到重叠点）；`legend(off)` 关闭图例（颜色含义在讲义文字中说明）；坐标轴标题用中文标注。


In [ ]:
%%stata
twoway (scatter log_waste_water log_gdp_per_capita [fw=v86] if east, msize(tiny) mcolor(blue%30)) (scatter log_waste_water log_gdp_per_capita [fw=v86] if middle, msize(tiny) mcolor(green%30)) (scatter log_waste_water log_gdp_per_capita [fw=v86] if west, msize(tiny) mcolor(red%30)), legend(off) xtitle("对数人均GDP") ytitle("对数工业废水排放量")
